# WEIGHT1 재개 — 충돌 진단부터 안전하게 진행

기존 학습은 **준비 구간의 미확인 충돌**로 멈췄습니다. 이 노트북은 같은 입력으로 준비 구간을 재현하고 차량 궤적을 저장합니다. 진단 자료를 검토하기 전에는 학습을 재개하지 않습니다.

1. 새 ZIP `FutureTraffic_WEIGHT1_Resume.zip`을 Drive의 `MyDrive/FutureTraffic`에 둡니다.
2. 기존 `explore_weight_v1`과 `explore_weight_long_v1` 폴더는 그대로 둡니다.
3. Colab에서 **런타임 → 모두 실행**을 누릅니다. 새 진단·재개 기록은 `explore_weight_resume_v1`에 저장합니다.

**처음 모두 실행하면 충돌 진단만 끝내고 ‘진단 검토 대기’로 멈출 수 있습니다. 정상이며, 아직 학습을 시작한 것이 아닙니다.** 진단을 마치면 이 대화에 “진단 끝났어”라고 알려주세요. 보고서는 Drive에 저장돼 있어 다시 올릴 필요가 없습니다. 충돌 검출만으로 예외를 추가하거나 실패한 에피소드를 건너뛰지 않습니다. 원인 검토와 별도 재개 승인이 준비된 경우에만 저장된 학습을 이어 갑니다. 기존 목표와 계산 장부는 보존하며, 맥에서는 시뮬레이션·학습을 실행하지 않습니다.

로그 칸을 중지하면 이 노트북이 시작한 실행기와 하위 작업을 함께 정리합니다. Colab 제공 자원과 연결 시간은 보장되지 않습니다.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE_ROOT / "FutureTraffic_WEIGHT1_Resume.zip"
SOURCE = DRIVE_ROOT / "explore_weight_v1"
STUDY = DRIVE_ROOT / "explore_weight_long_v1"
WORKSPACE = Path("/content/FutureTraffic_WEIGHT1_Resume")
RESULTS = DRIVE_ROOT / "explore_weight_resume_v1"
WORK = Path("/content/weight_resume_work")
assert ARCHIVE.is_file(), f"새 Resume ZIP이 없습니다: {ARCHIVE}"
for folder in (SOURCE, STUDY):
    assert folder.is_dir() and (folder / "experiment.json").is_file(), f"기존 결과 폴더가 없습니다: {folder}"
print("원본 실험:", SOURCE)
print("저장된 긴 학습:", STUDY)
print("새 진단·재개 기록:", RESULTS)


In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
# 기존 디렉터리를 지우지 않고 ZIP에 포함된 파일만 교체합니다.
import hashlib
import json
import stat
import sys
import types
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "WEIGHT1_RESUME", "WEIGHT1_RESUME 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "기존 모델은 Drive 원본에서 읽어야 합니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("weight1_resume_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE}")


In [ ]:
import os
import subprocess
import importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}
source_packages = json.loads((SOURCE / "experiment.json").read_text())["packages"]
assert all(source_packages.get(name) == version for name, version in EXPECTED.items()), "출발 실험의 패키지 버전이 이 Resume 묶음과 다릅니다."

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("출발 실험과 패키지 버전이 모두 같아 재설치를 건너뜁니다.")
else:
    print("출발 실험과 같은 패키지 버전을 준비합니다:", before)
    # SUMO-RL은 requirements의 지정 Git commit을 유지합니다.
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 출발 실험과 다릅니다: {after}"
print("실행 패키지 검증 통과:", after)


In [ ]:
# 실제 SUMO를 실행하지 않는 필수 검사부터 확인합니다.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_weight*.py", "-v"],
               cwd=WORKSPACE, check=True)
COMMON = ["--source", str(SOURCE), "--study", str(STUDY), "--out", str(RESULTS), "--work", str(WORK),
          "--workers", "auto", "--wall-hours", "24", "--compute-hours", "96"]
print("검사 통과. 다음 칸은 준비 구간의 충돌을 재현합니다. 아직 학습을 시작하지 않습니다.")


In [ ]:
# 진단 작업을 별도 프로세스로 실행합니다. 같은 결과 폴더의 중복 실행은 피합니다.
import datetime
sys.path.insert(0, str(WORKSPACE / "src"))
from explore_weight_resume_package import stop_owned_process_tree

RESULTS.mkdir(parents=True, exist_ok=True)
LOG = RESULTS / "colab_stdout.log"
PROCESS_RECORD = RESULTS / "colab_process.json"
RESUME_PROCESS = None
CURRENT_COMMAND = None

def owned_resume_record():
    if not PROCESS_RECORD.is_file():
        return None
    try:
        record = json.loads(PROCESS_RECORD.read_text())
        pid = int(record["pid"])
        arguments = Path(f"/proc/{pid}/cmdline").read_bytes().rstrip(b"\0").split(b"\0")
        arguments = [value.decode() for value in arguments]
        status = Path(f"/proc/{pid}/status").read_text()
        script_matches = any(Path(value).name == "explore_weight_resume.py" for value in arguments)
        output_matches = Path(arguments[arguments.index("--out")+1]).absolute() == RESULTS.absolute()
        stat_fields = Path(f"/proc/{pid}/stat").read_text().rpartition(")")[2].split()
        identity_matches = int(stat_fields[19]) == int(record["started_tick"])
        if script_matches and output_matches and identity_matches and "State:\tZ" not in status:
            return record
    except (OSError, ValueError, KeyError, IndexError):
        pass
    return None

def launch_resume_command(command):
    global RESUME_PROCESS, CURRENT_COMMAND
    if command not in ("diagnose", "run"):
        raise ValueError("허용되지 않은 실행 단계입니다")
    RESUME_PROCESS = None
    CURRENT_COMMAND = command
    try:
        with LOG.open("ab", buffering=0) as stream:
            RESUME_PROCESS = subprocess.Popen([sys.executable, "-u", "src/explore_weight_resume.py", command, *COMMON],
                                              cwd=WORKSPACE, stdout=stream, stderr=subprocess.STDOUT,
                                              env={**os.environ, "PYTHONUNBUFFERED": "1"}, start_new_session=True)
        stat_fields = Path(f"/proc/{RESUME_PROCESS.pid}/stat").read_text().rpartition(")")[2].split()
        PROCESS_RECORD.write_text(json.dumps({"pid": RESUME_PROCESS.pid, "command": command,
            "started_tick": int(stat_fields[19]), "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "log": str(LOG)}, indent=2))
    except BaseException:
        if RESUME_PROCESS is not None and RESUME_PROCESS.poll() is None:
            stop_owned_process_tree(RESUME_PROCESS.pid, RESULTS, graceful_seconds=30, terminate_seconds=5)
        raise
    print("실행 단계:", command, "PID:", RESUME_PROCESS.pid)

existing = owned_resume_record()
if existing is not None:
    CURRENT_COMMAND = existing["command"]
    print("이미 실행 중인 진단·재개 작업의 로그로 연결합니다. PID:", existing["pid"])
else:
    launch_resume_command("diagnose")
print("실시간 기록:", LOG)


In [ ]:
# 진단 후 승인 여부를 확인합니다. 미승인 상태에서는 학습하지 않고 검토 대기로 끝납니다.
import time
from IPython.display import clear_output

try:
    while True:
        returncode = RESUME_PROCESS.poll() if RESUME_PROCESS is not None else None
        active = owned_resume_record()
        if LOG.is_file():
            with LOG.open("rb") as stream:
                stream.seek(max(0, LOG.stat().st_size-14000))
                recent = stream.read().decode("utf-8", errors="replace")
        else:
            recent = "아직 로그가 없습니다."
        clear_output(wait=True)
        print(recent)
        if active is None or (RESUME_PROCESS is not None and returncode is not None):
            if CURRENT_COMMAND == "diagnose" and returncode == 0:
                # run 명령 자체가 진단 검토·승인 없이는 학습을 거부합니다.
                print("진단 완료. 검토·승인 상태를 확인합니다.")
                launch_resume_command("run")
                continue
            if returncode == 78:
                print("진단 검토 대기 — 정상입니다. 아직 학습을 시작하지 않았습니다.")
                print("이 대화에 ‘진단 끝났어’라고 알려주세요. 보고서는 Drive에 저장돼 있습니다.")
            elif returncode not in (None, 0):
                print("실행 단계가 오류로 중단됐습니다. 위 로그와 아래 요약을 확인하세요. 종료 코드:", returncode)
            elif returncode is None:
                print("기존 프로세스가 종료됐습니다. 요약에서 실제 상태를 확인하세요.")
            else:
                print("실행 단계가 끝났습니다. 아래 요약을 확인하세요.")
            break
        print("실행 중 — 30초마다 최근 로그를 갱신합니다. 결과:", RESULTS)
        time.sleep(30)
except KeyboardInterrupt:
    active = owned_resume_record()
    if active is not None:
        print("저장·정산과 소유 작업 종료를 기다립니다.")
        try:
            stopped = stop_owned_process_tree(int(active["pid"]), RESULTS, graceful_seconds=30, terminate_seconds=5)
        except KeyboardInterrupt:
            stopped = stop_owned_process_tree(int(active["pid"]), RESULTS, graceful_seconds=0, terminate_seconds=0)
        if RESUME_PROCESS is not None:
            try:
                RESUME_PROCESS.wait(timeout=3)
            except subprocess.TimeoutExpired:
                pass
        print("진단·재개 실행기와 하위 작업 중단:", stopped)
    else:
        print("실행 중인 작업이 없습니다.")


In [ ]:
from IPython.display import Markdown, display

subprocess.run([sys.executable, "src/explore_weight_resume.py", "summary", *COMMON], cwd=WORKSPACE, check=True)
summary_file = RESULTS / "summary.md"
if summary_file.is_file():
    display(Markdown(summary_file.read_text()))
else:
    print("요약 파일이 없습니다. 실행 기록을 확인하세요:", LOG)
print("원본 학습:", STUDY)
print("새 진단·재개 자료:", RESULTS)
